In [1]:
import pyfaidx
from pyfaidx import Fasta
from pathlib import Path

from torch.utils.data import DataLoader, Dataset

from data import EnformerSignalDataset


from finetune_enformer import EnformerFinetune, EnformerFineTunerPL

from enformer_pytorch import Enformer

from tqdm import tqdm

from pyfaidx import Fasta

/public/home/zju12218076/miniconda3/envs/project/lib/python3.10/site-packages/sorted_nearest/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/public/home/zju12218076/miniconda3/envs/project/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
fasta = Fasta("./data/hg38.fa")



In [3]:
# traning data
train_chr = ['chr3', 'chr4', 'chr5', 'chr6', 'chr7',
            'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17',
            'chr19', 'chr21', 'chr22', 'chrX']


# validation data

validation_chr = ['chr2', 'chr18', 'chr20']

# test data

test_chr = ['chr1', 'chr8']

In [4]:
# 生成训练，验证和测试数据集合的区间文件

def create_genome_tiles(chroms, output_bed, window_size= 196_608, stride= 196_608 - 320 * 128, fasta_file="./data/hg38.fa"):
    fasta = Fasta(fasta_file)
    
    if isinstance(chroms, str):
        chroms = [chroms]

    with open(output_bed, 'w') as f_out:
        for chrom in chroms:
            length = len(fasta[chrom])
            # print(f"{chrom} {length}")

            for start in range(100, length-window_size, stride):
                end = start + window_size
                f_out.write(f"{chrom}\t{start}\t{end}\n")

# 训练数据
create_genome_tiles(chroms=train_chr, output_bed="./develop_test/train_3d_region.bed")

# 验证数据
create_genome_tiles(chroms=validation_chr, output_bed="./develop_test/validation_3d_region.bed")

# 测试数据集合
create_genome_tiles(chroms=test_chr, output_bed="./develop_test/test_3d_region.bed")

In [5]:
# 表观数据
bigwig_file = "/public/home/zju12218076/o_D"
bw_file = [
    '786O_WT_CS_H3K27ac_r1.bw', '786O_WT_CS_H3K4me1_r1.bw',
    '786O_WT_CS_H3K4me3_r1.bw', '786O_WT_CS_CTCF_r2.bw',
    '786O_WT_ATAC_r1.bw'
]

bw_file = [str(Path(bigwig_file, f)) for f in bw_file]


In [6]:
# EnformerSignalDataset(interval_file="./develop_test/train_region.bed", fasta_file="./data/hg38.fa", bigwig_file=bw_file)

In [7]:
#pytorch_lightning 数据

import pytorch_lightning as pl
from torch.utils.data import DataLoader, random_split
# 假设我们之前的 EnformerSignalDataset 类定义在这里
# from my_dataset import EnformerSignalDataset 

class EnformerDataModule(pl.LightningDataModule):
    def __init__(self, intervals_file_train, intervals_file_validation, fasta_file, bigwig_files, 
                 batch_size=4, num_workers=2, val_split=0.1):
        super().__init__()
        self.intervals_file_train = intervals_file_train
        self.intervals_file_validation = intervals_file_validation
        self.fasta_file = fasta_file
        self.bigwig_files = bigwig_files
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.val_split = val_split
        
        # 保存超参数，方便后续访问和记录
        self.save_hyperparameters()

    def setup(self, stage=None):
        # 这个方法在训练/验证/测试开始时被调用，只在需要的进程上执行一次
        # 我们在这里创建完整的数据集并进行划分
        self.train_dataset = EnformerSignalDataset(self.intervals_file_train, 
                                             self.fasta_file, 
                                             self.bigwig_files)
        
        self.validation_dataset = EnformerSignalDataset(self.intervals_file_validation,
                                                   self.fasta_file,
                                                   self.bigwig_files)
        
        

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, 
                          shuffle=True, num_workers=self.num_workers, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.validation_dataset, batch_size=self.batch_size, 
                        shuffle=False, num_workers=self.num_workers, pin_memory=True)


In [8]:
data = EnformerDataModule(intervals_file_train="./develop_test/train_region.bed",
                  intervals_file_validation="./develop_test/validation_region.bed",
                  fasta_file="./data/hg38.fa",
                  bigwig_files=bw_file, batch_size=2, num_workers=1)

In [9]:
data.setup()
data.val_dataloader()

In [17]:
s, t = next(iter(data.val_dataloader()))

torch.Size([4, 896, 5])

In [9]:
enform = Enformer.from_pretrained('EleutherAI/enformer-official-rough', use_tf_gamma=True)

In [10]:
# pl_model = EnformerFinetune(pretrained_enformer=enform, num_chip_tracks=5)
pl_model = EnformerFineTunerPL(enform)

/public/home/zju12218076/miniconda3/envs/project/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'enformer_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['enformer_model'])`.


In [11]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

In [12]:
checkpoint_callback = ModelCheckpoint(
    dirpath='./checkpoints/',
    filename='enformer-finetuned-{epoch:02d}-{val_pearson_mean:.4f}',
    monitor='val_pearson_mean',
    mode='max',
    save_top_k=1
)

early_stopping_callback = EarlyStopping(
    monitor='val_pearson_mean',
    patience=3,
    mode='max'
)

In [13]:
trainer = Trainer(
    accelerator='gpu',  # 自动使用 GPU
    devices=[2, 3, 4],         # 使用所有可用的 GPU
    # strategy='ddp',     # 使用分布式数据并行 (如果有多于1个GPU)
    max_epochs=20,
    callbacks=[checkpoint_callback, early_stopping_callback],
    # precision='16-mixed' # 使用混合精度训练，可以加速训练并节省显存
)

/public/home/zju12218076/miniconda3/envs/project/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /public/home/zju12218076/miniconda3/envs/project/lib ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/public/home/zju12218076/miniconda3/envs/project/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of th

In [24]:
pl_model = pl_model.cpu()

In [25]:
trainer.fit(pl_model, datamodule=data)

RuntimeError: Lightning can't create new processes if CUDA is already initialized. Did you manually call `torch.cuda.*` functions, have moved the model to the device, or allocated memory on the GPU any other way? Please remove any such calls, or change the selected strategy. You will have to restart the Python kernel.